# หัวข้อใหญ่ 6 — Audio / Speech (เสียง)

โจทย์ตัวอย่าง: จำแนกเสียง (ชนิดเสียง/คำสั่งเสียง/อารมณ์), จำแนกประเภทดนตรี, ตรวจจับเหตุการณ์เสียง
(ถ้าเป็น "ถอดเสียงเป็นข้อความ" = ASR ดูหมายเหตุท้ายโน้ตบุ๊ก -> ใช้ Whisper)

วิธีในโน้ตบุ๊กนี้ (จำแนกเสียง):
- วิธีที่ 1 (ง่ายสุด) = สกัดฟีเจอร์เสียง (MFCC/spectral) ด้วย `librosa` แล้วโยนเข้า `AutoGluon` เหมือนตาราง
- วิธีที่ 2 (ไม่บังคับ) = ฟีเจอร์เดิม + `LightGBM`
- วิธีที่ 3 (ขั้นสูง) = แปลงเป็น `mel-spectrogram` (รูป) แล้วใช้โน้ตบุ๊ก 01 image_classification


## เมื่อไหร่ใช้โน้ตบุ๊กนี้

ใช้เมื่อ input เป็นไฟล์เสียง (.wav/.mp3/...) และตอบ "1 คลาสต่อ 1 ไฟล์"
ถ้าต้อง "ถอดเสียงเป็นข้อความ" (ASR) -> ใช้ Whisper (หมายเหตุท้ายไฟล์)

ต้องแก้ (`# << แก้`): ชื่อ competition, โฟลเดอร์เสียง, `FILE_COL` (คอลัมน์ชื่อไฟล์), `LABEL_COL`, `SR`

## ขั้นที่ 1 — ติดตั้ง

In [ ]:
!pip -q install librosa soundfile pandas numpy scikit-learn lightgbm
!pip -q install -U "autogluon.tabular[all]"   # วิธีที่ 1

## ขั้นที่ 2 — เอาข้อมูลเข้า (เลือก 1 ใน 3 วิธี) + เช็ค GPU

เซลล์ล่างรองรับ 3 วิธี แก้แค่ตัวแปรบนสุดให้ตรงกับวิธีที่ใช้:

วิธี A (แนะนำ) ดาวน์โหลดจาก Kaggle อัตโนมัติ: ต้องมี `kaggle.json` (kaggle.com -> รูปโปรไฟล์ -> Settings -> Account -> Create New Token)
- บน `Kaggle Notebook`: ข้อมูลต่อไว้ที่ `/kaggle/input/...` แล้ว ไม่ต้องใส่ token
- บน `Colab/เครื่อง`: ใส่ `KAGGLE_USERNAME` + `KAGGLE_KEY`

วิธี B โหลดข้อมูลมาเครื่องตัวเองแล้วอัปขึ้น Colab เอง: ลากไฟล์ (เช่น `data.zip`) ไปวางในแถบ Files ซ้ายมือของ Colab
แล้วตั้ง `DATA_DIR = "/content"` (หรือโฟลเดอร์ที่วาง) -> เซลล์จะแตก zip ให้เอง ไม่ต้องใช้ token

วิธี C ต่อ Google Drive: รัน `from google.colab import drive; drive.mount('/content/drive')` ก่อน แล้วตั้ง `DATA_DIR = "/content/drive/MyDrive/โฟลเดอร์ของคุณ"`

เซลล์นี้เช็ค GPU ให้ด้วย ถ้างานเป็น deep learning (รูป/ข้อความ/สัญญาณดิบ) ควรเปิด GPU: เมนู `Runtime > Change runtime type > T4 GPU`

In [ ]:
import os, glob, subprocess

# ----- วิธี A: ดาวน์โหลดจาก Kaggle -----
COMP = "ใส่-slug-ของ-competition-audio-ตรงนี้"      # << แก้: slug ท้าย URL เช่น kaggle.com/competitions/titanic -> "titanic"  (ใช้เฉพาะวิธี A)
KAGGLE_USERNAME = ""   # << แก้ (วิธี A, เฉพาะ Colab/เครื่อง) เช่น "peetwan1"  (บน Kaggle Notebook เว้นว่างได้)
KAGGLE_KEY      = ""   # << แก้ (วิธี A, เฉพาะ Colab/เครื่อง) เช่น "0a1b2c3d4e5f..." (คีย์ยาว ~32 ตัวจาก kaggle.json)
# ----- วิธี B/C: อัปโหลดเอง / ต่อ Google Drive -----
DATA_DIR = ""          # << แก้: ถ้าอัปโหลดข้อมูลเอง ใส่ path โฟลเดอร์ เช่น "/content" (วิธี B) หรือ "/content/drive/MyDrive/data" (วิธี C) ; ใช้ Kaggle (วิธี A) ปล่อยว่าง

# เช็ค GPU (ถ้าไม่มี + งานเป็น deep learning -> เปิดที่เมนู Runtime > Change runtime type > T4 GPU)
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ไม่มี (งาน deep learning จะช้า; งานตาราง/ตัดคำ ไม่ต้องใช้ GPU ก็ได้)")
except Exception:
    pass

def get_data(comp):
    # วิธี B/C: ผู้ใช้ตั้ง DATA_DIR เอง (อัปโหลด/ต่อ Drive) -- แตก zip ในโฟลเดอร์นั้นให้ด้วยถ้ามี
    if DATA_DIR and os.path.isdir(DATA_DIR):
        for z in glob.glob(os.path.join(DATA_DIR, "*.zip")):
            subprocess.run(["unzip", "-o", "-q", z, "-d", DATA_DIR])
        print("ใช้ข้อมูลจากโฟลเดอร์ที่ตั้งเอง:", DATA_DIR); return DATA_DIR
    # บน Kaggle Notebook ข้อมูลต่อไว้แล้ว
    kpath = f"/kaggle/input/{comp}"
    if os.path.isdir(kpath):
        print("อยู่บน Kaggle ข้อมูลอยู่ที่", kpath); return kpath
    # วิธี A: ดาวน์โหลดด้วย Kaggle API
    if KAGGLE_USERNAME and KAGGLE_KEY:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        open(os.path.expanduser("~/.kaggle/kaggle.json"), "w").write(
            '{"username":"%s","key":"%s"}' % (KAGGLE_USERNAME, KAGGLE_KEY))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    os.makedirs("data", exist_ok=True)
    subprocess.run(["kaggle","competitions","download","-c",comp,"-p","data"], check=True)
    for z in glob.glob("data/*.zip"):
        subprocess.run(["unzip","-o","-q",z,"-d","data"], check=True)
    return "data"

DATA_DIR = get_data(COMP)
print("\nไฟล์ที่มี (ดูชื่อไฟล์/โฟลเดอร์ แล้วแก้ในเซลล์ถัดไปให้ตรง):")
for p in sorted(glob.glob(os.path.join(DATA_DIR, "**"), recursive=True))[:40]:
    print("  ", p)

## ขั้นที่ 3 — โหลดข้อมูล + CONFIG

In [ ]:
import os, glob, pandas as pd, numpy as np
AUDIO_EXT = (".wav",".mp3",".flac",".ogg",".m4a")
def find(name):
    h=glob.glob(os.path.join(DATA_DIR,"**",name),recursive=True); return h[0] if h else None
def audio_dir(keyword):
    cand=[d for d,_,fs in os.walk(DATA_DIR) if keyword in d.lower() and any(f.lower().endswith(AUDIO_EXT) for f in fs)]
    return max(cand,key=lambda d:len(os.listdir(d))) if cand else None

TRAIN_CSV = find("train.csv"); SAMPLE_SUB = find("sample_submission.csv")
TRAIN_AUD = audio_dir("train")   # << แก้ถ้าผิด เช่น os.path.join(DATA_DIR,"audio/train")
TEST_AUD  = audio_dir("test")
SR        = 16000      # << แก้: sample rate ที่จะอ่าน (16000 พอสำหรับเสียงพูด, เพลงใช้ 22050)
df = pd.read_csv(TRAIN_CSV); sample = pd.read_csv(SAMPLE_SUB)
ID_COL, ANS_COL = sample.columns[0], sample.columns[1]
print("train คอลัมน์:", list(df.columns)); print("เสียง train:", TRAIN_AUD, "| test:", TEST_AUD)
display(df.head()); display(sample.head())
FILE_COL  = "filename"   # << แก้: คอลัมน์ชื่อไฟล์เสียงใน train.csv เช่น "fname", "audio", "id"
LABEL_COL = "label"      # << แก้: คอลัมน์คลาส เช่น "label", "category", "target"
TEST_EXT  = ".wav"       # << แก้: นามสกุลไฟล์เสียง test (ถ้าใน sample id มีนามสกุลแล้ว ใส่ "")

## 🔎 โจทย์นี้ต้องส่งอะไร + วัดด้วยอะไร (รันก่อนทำ submission)

เซลล์นี้ตอบ 3 คำถามที่มือใหม่มักไม่รู้:
- ต้องเติม "คอลัมน์อะไร" ลง submission และเป็น "ชนิดไหน" (ดูจาก sample_submission)
- โจทย์วัดด้วย "metric อะไร" (ดึงจาก Kaggle ให้อัตโนมัติ)
- ต้องส่งเป็น "ป้าย / ความน่าจะเป็น / ตัวเลข" แบบไหน

In [ ]:
# บอกว่าต้องเติมอะไรลง submission + ตัวอย่างค่าที่ sample ให้มา
print("ไฟล์ส่งต้องมีคอลัมน์:", list(sample.columns), "| จำนวนแถว:", len(sample))
for _c in list(sample.columns)[1:]:
    print(f"  - เติม '{_c}': ชนิด {sample[_c].dtype}, ตัวอย่างค่าใน sample = {list(sample[_c].dropna().unique()[:3])}")
# ดึง metric จาก Kaggle อัตโนมัติ (ถ้าต่อ API ได้)
_metric = None
try:
    from kaggle.api.kaggle_api_extended import KaggleApi
    _api = KaggleApi(); _api.authenticate()
    _resp = _api.competitions_list(search=COMP)
    for _co in (getattr(_resp, "competitions", _resp) or []):   # kaggle ใหม่คืน response (ใช้ .competitions), เก่าคืน list
        if str(getattr(_co, "ref", "")).rstrip("/").split("/")[-1] == COMP:
            _metric = getattr(_co, "evaluation_metric", None) or getattr(_co, "evaluationMetric", None); break
except Exception:
    pass
def _how_to_send(m):
    m = (m or "").lower()
    if any(k in m for k in ["auc","roc","log loss","logloss","brier","probab"]): return "ความน่าจะเป็น (เลข 0-1)"
    if any(k in m for k in ["rmse","mae","mse","r2","rmsle","error","mape","smape"]): return "ตัวเลขต่อเนื่อง"
    return "ป้าย/คลาส (เช่น 0/1 หรือชื่อคลาส)"
print("\nMetric (ดึงจาก Kaggle):", _metric or "ดึงไม่ได้ -> เปิด tab Evaluation บนหน้า Kaggle อ่านเอง")
print("=> ปกติต้องส่งเป็น:", _how_to_send(_metric), "(เช็คกับ tab Evaluation อีกที)")

## สกัดฟีเจอร์เสียงต่อไฟล์ (ใช้ทั้งวิธีที่ 1 และ 2)

ฟีเจอร์มาตรฐาน: MFCC (รูปร่างของเสียง) + spectral + zero-crossing + พลังงาน -> เฉลี่ย/ส่วนเบี่ยงเบนต่อไฟล์

In [ ]:
import librosa
def audio_features(path, sr=SR):
    y,_ = librosa.load(path, sr=sr, mono=True)
    if len(y) < sr: y = np.pad(y, (0, sr-len(y)))   # กันไฟล์สั้นเกิน
    f = {}
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    for i in range(20): f[f"mfcc{i}_m"]=mfcc[i].mean(); f[f"mfcc{i}_s"]=mfcc[i].std()
    f["cent"]=librosa.feature.spectral_centroid(y=y,sr=sr).mean()
    f["bw"]=librosa.feature.spectral_bandwidth(y=y,sr=sr).mean()
    f["roll"]=librosa.feature.spectral_rolloff(y=y,sr=sr).mean()
    f["zcr"]=librosa.feature.zero_crossing_rate(y).mean()
    f["rms"]=librosa.feature.rms(y=y).mean()
    chroma=librosa.feature.chroma_stft(y=y,sr=sr);
    for i in range(12): f[f"chroma{i}"]=chroma[i].mean()
    return f
def build(file_list, audio_dir):
    rows=[]
    for fn in file_list:
        p=os.path.join(audio_dir, str(fn))
        try: rows.append(audio_features(p))
        except Exception: rows.append({})
    return pd.DataFrame(rows).fillna(0)
Xtr = build(df[FILE_COL].tolist(), TRAIN_AUD)
Xte = build([str(i)+TEST_EXT for i in sample[ID_COL].tolist()], TEST_AUD)
print("ฟีเจอร์ต่อไฟล์:", Xtr.shape[1], "ตัว")

## วิธีที่ 1 — ฟีเจอร์ + AutoGluon (ง่ายสุด)

In [ ]:
from autogluon.tabular import TabularPredictor
ag = Xtr.copy(); ag[LABEL_COL] = df[LABEL_COL].values
pred = TabularPredictor(label=LABEL_COL, path="ag_audio").fit(ag, presets="best_quality", time_limit=600)   # << แก้ time_limit
out = sample.copy(); out[ANS_COL] = pred.predict(Xte).values
out.to_csv("submission.csv", index=False); print("บันทึก submission.csv"); display(out.head())

## วิธีที่ 2 — ฟีเจอร์ + LightGBM + CV (ไม่บังคับ)

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
le = LabelEncoder(); y = le.fit_transform(df[LABEL_COL])
oof = np.zeros(len(y)); pred = np.zeros((len(Xte), len(le.classes_)))
for tr,va in StratifiedKFold(5,shuffle=True,random_state=42).split(Xtr,y):
    m=lgb.LGBMClassifier(n_estimators=800,learning_rate=0.03,random_state=42,verbose=-1)
    m.fit(Xtr.iloc[tr],y[tr],eval_set=[(Xtr.iloc[va],y[va])],callbacks=[lgb.early_stopping(50,verbose=False)])
    oof[va]=m.predict(Xtr.iloc[va]); pred+=m.predict_proba(Xte)/5
print("OOF macro-F1 =", round(f1_score(y,oof,average="macro"),4))
out=sample.copy(); out[ANS_COL]=le.inverse_transform(pred.argmax(1))
out.to_csv("submission_lgbm.csv",index=False); print("บันทึก submission_lgbm.csv")

## หมายเหตุ — ถ้าเป็น ASR (ถอดเสียงเป็นข้อความ)

ใช้ Whisper สำเร็จรูป (zero-shot, รองรับไทย):
```python
!pip install -U openai-whisper
import whisper
m = whisper.load_model("small")   # tiny/base/small/medium/large
text = m.transcribe("ไฟล์.wav", language="th")["text"]
```
แล้วเขียน text ลง submission ตามฟอร์แมต (วัดด้วย CER/WER -> ใช้ pythainlp.util.word_error_rate วัดในเครื่อง)

## ขั้นสุดท้าย — ส่ง submission ขึ้น Kaggle

เช็คก่อนส่งทุกครั้ง: ไฟล์ submission ต้องมีชื่อคอลัมน์ + จำนวนแถว ตรงกับ `sample_submission.csv` เป๊ะ ๆ
- บน Kaggle Notebook: กดปุ่ม `Submit` ที่แถบขวา (ง่ายสุด) หรือใช้คำสั่งข้างล่าง
- บน Colab/เครื่อง: รันเซลล์ข้างล่าง (ต้องตั้ง token แล้ว)

In [ ]:
import pandas as pd, glob, os
FILE = "submission.csv"   # << แก้เป็นไฟล์ที่จะส่ง เช่น "submission_lgbm.csv" หรือ "submission_timm.csv"
# ตรวจรูปแบบก่อนส่งอัตโนมัติ (กันแก้ config ผิดแล้วส่งฟอร์แมตเพี้ยน -> ได้ 0 คะแนน)
_sub = pd.read_csv(FILE)
_sam = glob.glob(os.path.join(DATA_DIR, "*ample*ubmission*.csv"))
if _sam:
    _s = pd.read_csv(_sam[0])
    print("คอลัมน์ตรง sample:", list(_sub.columns)==list(_s.columns), "| จำนวนแถวตรง:", len(_sub)==len(_s))
    assert list(_sub.columns)==list(_s.columns), f"คอลัมน์ไม่ตรง sample! {list(_sub.columns)} != {list(_s.columns)} -> แก้ก่อนส่ง"
print("พร้อมส่ง:", FILE, _sub.shape)
!kaggle competitions submit -c {COMP} -f {FILE} -m "audio features autogluon"
!kaggle competitions submissions -c {COMP} | head